# Task5 Rescued Cases Interactive Explorer (Gemma)

This notebook visualizes and explores rescued samples from:
- output/supplementary_figures/task5_top50_rescued_cases_gemma.csv
- output/supplementary_figures/task5_all_low_fail_high_success_cases_gemma.csv (for full response text)

Features:
1. Filter by prompt variant, rescue type, gold winner, human winner, model pair, and keyword.
2. Click points in scatter plot to select sample.
3. Browse one sample at a time with full details (prompt setting, verdict trajectory, response A/B, reference).

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from html import escape
from IPython.display import display, HTML, Markdown

BASE_DIR = Path('..')
TOP50_PATH = BASE_DIR / 'output' / 'supplementary_figures' / 'task5_top50_rescued_cases_gemma.csv'
ALL_PATH = BASE_DIR / 'output' / 'supplementary_figures' / 'task5_all_low_fail_high_success_cases_gemma.csv'

if not TOP50_PATH.exists():
    raise FileNotFoundError(f'Missing file: {TOP50_PATH}')

df_top = pd.read_csv(TOP50_PATH)
df = df_top.copy()

if ALL_PATH.exists():
    df_all = pd.read_csv(ALL_PATH)
    keep_cols = [c for c in ['question_id', 'prompt_variant', 'answer_a_text', 'answer_b_text', 'reference_answer'] if c in df_all.columns]
    if len(keep_cols) >= 3:
        df = df.merge(df_all[keep_cols].drop_duplicates(subset=['question_id', 'prompt_variant']), on=['question_id', 'prompt_variant'], how='left')

def _pick_col(full_col, snippet_col):
    if full_col in df.columns:
        base = df[full_col]
    elif snippet_col in df.columns:
        base = df[snippet_col]
    else:
        base = pd.Series([''] * len(df), index=df.index)
    return base.fillna('')

df['answer_a_show'] = _pick_col('answer_a_text', 'answer_a_text_snippet').astype(str)
df['answer_b_show'] = _pick_col('answer_b_text', 'answer_b_text_snippet').astype(str)
df['reference_show'] = _pick_col('reference_answer', 'reference_answer_snippet').astype(str)

df['rescue_type'] = np.select(
    [
        df['rescued_ensemble_only'].astype(bool),
        df['rescued_by_single'].astype(bool) & df['rescued_by_ensemble'].astype(bool),
        df['rescued_by_single'].astype(bool),
        df['rescued_by_ensemble'].astype(bool),
    ],
    [
        'Ensemble Only',
        'Both Single and Ensemble',
        'Single Only',
        'Ensemble (possibly overlap)',
    ],
    default='Other',
)

df['sample_id'] = np.arange(len(df))
df['keyword_blob'] = (
    df['question_id'].astype(str) + ' ' +
    df['prompt_variant'].astype(str) + ' ' +
    df['model_a'].astype(str) + ' ' +
    df['model_b'].astype(str) + ' ' +
    df['answer_a_show'].astype(str) + ' ' +
    df['answer_b_show'].astype(str) + ' ' +
    df['reference_show'].astype(str)
).str.lower()

print(f'Loaded samples: {len(df):,}')
print('Columns:', df.columns.tolist())

Loaded samples: 38
Columns: ['question_id', 'prompt_variant', 'gold_winner', 'human_winner', 'v_low_sp', 'v_high_sp_1.5', 'v_high_sp_2.0', 'v_high_sp_3.0', 'v_high_ens_1.5', 'v_high_ens_2.0', 'v_high_ens_3.0', 'rescued_by_single', 'rescued_by_ensemble', 'rescued_ensemble_only', 'ent_high_ens_mean', 'distinct_high_ens_mean', 'model_a', 'model_b', 'answer_a_text_snippet', 'answer_b_text_snippet', 'reference_answer_snippet', 'answer_a_text', 'answer_b_text', 'reference_answer', 'answer_a_show', 'answer_b_show', 'reference_show', 'rescue_type', 'sample_id', 'keyword_blob']


In [3]:
overview = pd.DataFrame({
    'Metric': [
        'N samples',
        'N prompt variants',
        'N rescue types',
        'Mean entropy (high ensemble)',
        'Mean distinct verdicts (high ensemble)',
    ],
    'Value': [
        len(df),
        df['prompt_variant'].nunique(),
        df['rescue_type'].nunique(),
        df['ent_high_ens_mean'].mean(),
        df['distinct_high_ens_mean'].mean(),
    ],
})
display(overview)

fig1 = px.histogram(df, x='ent_high_ens_mean', color='prompt_variant', barmode='overlay', nbins=20,
                    title='Entropy Distribution (High Ensemble)')
fig1.update_layout(height=380)
fig1.show()

fig2 = px.bar(df['rescue_type'].value_counts().reset_index(), x='rescue_type', y='count',
              title='Rescue Type Counts')
fig2.update_layout(height=380, xaxis_title='Rescue Type', yaxis_title='Count')
fig2.show()

,Metric,Value
0,N samples,38.000000
1,N prompt variants,2.000000
2,N rescue types,3.000000
3,Mean entropy (high ensemble),0.819671
4,Mean distinct verdicts (high ensemble),2.175439


In [4]:
prompt_opts = sorted(df['prompt_variant'].dropna().unique().tolist())
rescue_opts = sorted(df['rescue_type'].dropna().unique().tolist())
gold_opts = ['ALL'] + sorted(df['gold_winner'].dropna().astype(str).unique().tolist())
human_opts = ['ALL'] + sorted(df['human_winner'].dropna().astype(str).unique().tolist())
model_opts = ['ALL'] + sorted((df['model_a'].astype(str) + ' vs ' + df['model_b'].astype(str)).unique().tolist())

w_prompt = widgets.SelectMultiple(options=prompt_opts, value=tuple(prompt_opts), description='Prompt', rows=3)
w_rescue = widgets.SelectMultiple(options=rescue_opts, value=tuple(rescue_opts), description='Rescue', rows=4)
w_gold = widgets.Dropdown(options=gold_opts, value='ALL', description='Gold')
w_human = widgets.Dropdown(options=human_opts, value='ALL', description='Human')
w_model = widgets.Dropdown(options=model_opts, value='ALL', description='Model Pair')
w_keyword = widgets.Text(value='', description='Keyword', placeholder='question_id or text keyword')
w_sort = widgets.Dropdown(options=['ent_high_ens_mean', 'distinct_high_ens_mean', 'question_id'], value='ent_high_ens_mean', description='Sort by')
w_desc = widgets.Checkbox(value=True, description='Descending')
w_idx = widgets.IntSlider(value=0, min=0, max=max(len(df)-1, 0), step=1, description='Sample')

out_status = widgets.Output()
out_scatter = widgets.Output()
out_table = widgets.Output()
out_detail = widgets.Output()

def _fmt_float(x):
    try:
        return f"{float(x):.4f}"
    except Exception:
        return 'NA'

def apply_filters():
    d = df.copy()
    d = d[d['prompt_variant'].isin(list(w_prompt.value))]
    d = d[d['rescue_type'].isin(list(w_rescue.value))]
    if w_gold.value != 'ALL':
        d = d[d['gold_winner'].astype(str) == w_gold.value]
    if w_human.value != 'ALL':
        d = d[d['human_winner'].astype(str) == w_human.value]
    if w_model.value != 'ALL':
        pair = d['model_a'].astype(str) + ' vs ' + d['model_b'].astype(str)
        d = d[pair == w_model.value]
    key = w_keyword.value.strip().lower()
    if key:
        d = d[d['keyword_blob'].str.contains(key, na=False)]
    d = d.sort_values(w_sort.value, ascending=not w_desc.value).reset_index(drop=True)
    return d

def render_sample(d):
    with out_status:
        out_status.clear_output(wait=True)
        print(f'Filtered samples: {len(d)}')

    if len(d) == 0:
        with out_scatter:
            out_scatter.clear_output(wait=True)
            print('No samples after filtering.')
        with out_table:
            out_table.clear_output(wait=True)
            print('No rows to display.')
        with out_detail:
            out_detail.clear_output(wait=True)
            print('No sample details.')
        w_idx.max = 0
        w_idx.value = 0
        return

    w_idx.max = len(d) - 1
    if w_idx.value > w_idx.max:
        w_idx.value = w_idx.max

    with out_scatter:
        out_scatter.clear_output(wait=True)
        fig_data = [
            go.Scatter(
                x=d['ent_high_ens_mean'],
                y=d['distinct_high_ens_mean'],
                mode='markers',
                marker=dict(size=10, opacity=0.8),
                text=('qid=' + d['question_id'].astype(str) + ', ' + d['prompt_variant'].astype(str)),
                customdata=np.arange(len(d)),
                hovertemplate='Entropy=%{x:.3f}<br>Distinct=%{y:.3f}<br>%{text}<extra></extra>',
            )
        ]
        fig_layout = dict(
            title='Click a point to select sample',
            xaxis_title='Mean Vote Entropy (High Ensemble)',
            yaxis_title='Mean Distinct Verdicts (High Ensemble)',
            height=420,
        )
        try:
            fig = go.FigureWidget(data=fig_data)
            fig.update_layout(**fig_layout)

            def _on_click(*args):
                if len(args) < 2:
                    return
                points = args[1]
                if points.point_inds:
                    w_idx.value = int(points.point_inds[0])

            fig.data[0].on_click(_on_click)
            display(fig)
        except Exception as e:
            fig = go.Figure(data=fig_data)
            fig.update_layout(**fig_layout)
            display(fig)
            print(f'Interactive click selection unavailable (FigureWidget issue): {e}')

    with out_table:
        out_table.clear_output(wait=True)
        cols = [
            'question_id', 'prompt_variant', 'rescue_type', 'gold_winner', 'human_winner',
            'v_low_sp', 'v_high_sp_1.5', 'v_high_sp_2.0', 'v_high_sp_3.0',
            'v_high_ens_1.5', 'v_high_ens_2.0', 'v_high_ens_3.0',
            'ent_high_ens_mean', 'distinct_high_ens_mean',
        ]
        show_cols = [c for c in cols if c in d.columns]
        display(d[show_cols].head(200))

    row = d.iloc[w_idx.value]
    verdict_tbl = pd.DataFrame({
        'Setting': ['Low SP', 'High SP 1.5', 'High SP 2.0', 'High SP 3.0', 'High ENS 1.5', 'High ENS 2.0', 'High ENS 3.0'],
        'Verdict': [
            row.get('v_low_sp', np.nan),
            row.get('v_high_sp_1.5', np.nan),
            row.get('v_high_sp_2.0', np.nan),
            row.get('v_high_sp_3.0', np.nan),
            row.get('v_high_ens_1.5', np.nan),
            row.get('v_high_ens_2.0', np.nan),
            row.get('v_high_ens_3.0', np.nan),
        ],
    })

    with out_detail:
        out_detail.clear_output(wait=True)
        qid = escape(str(row.get('question_id', 'NA')))
        prompt = escape(str(row.get('prompt_variant', 'NA')))
        rescue = escape(str(row.get('rescue_type', 'NA')))
        gold = escape(str(row.get('gold_winner', 'NA')))
        human = escape(str(row.get('human_winner', 'NA')))
        model_a = escape(str(row.get('model_a', 'NA')))
        model_b = escape(str(row.get('model_b', 'NA')))
        ent = _fmt_float(row.get('ent_high_ens_mean', np.nan))
        distinct = _fmt_float(row.get('distinct_high_ens_mean', np.nan))
        header = f"""
        <h3>Sample Detail: question_id={qid} | prompt={prompt}</h3>
        <b>Rescue Type:</b> {rescue}<br>
        <b>Gold Winner:</b> {gold} | <b>Human Winner:</b> {human}<br>
        <b>Model A:</b> {model_a}<br>
        <b>Model B:</b> {model_b}<br>
        <b>Entropy Mean:</b> {ent} | <b>Distinct Verdict Mean:</b> {distinct}
        """
        display(HTML(header))
        display(verdict_tbl)

        answer_a = escape(str(row.get('answer_a_show', '')))
        answer_b = escape(str(row.get('answer_b_show', '')))
        reference = escape(str(row.get('reference_show', '')))

        display(Markdown('### Response A'))
        display(HTML(f"<div style='white-space: pre-wrap; border: 1px solid #ddd; padding: 8px;'>{answer_a}</div>"))

        display(Markdown('### Response B'))
        display(HTML(f"<div style='white-space: pre-wrap; border: 1px solid #ddd; padding: 8px;'>{answer_b}</div>"))

        display(Markdown('### Reference / Gold Explanation'))
        display(HTML(f"<div style='white-space: pre-wrap; border: 1px solid #ddd; padding: 8px;'>{reference}</div>"))

def refresh(_=None):
    d = apply_filters()
    render_sample(d)

for w in [w_prompt, w_rescue, w_gold, w_human, w_model, w_keyword, w_sort, w_desc, w_idx]:
    w.observe(refresh, names='value')

control_box = widgets.VBox([
    widgets.HBox([w_prompt, w_rescue]),
    widgets.HBox([w_gold, w_human, w_model]),
    widgets.HBox([w_keyword, w_sort, w_desc]),
    w_idx,
    out_status,
])

display(control_box)
display(out_scatter)
display(out_table)
display(out_detail)

refresh()

Output()

Output()

Output()

In [13]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_distances

RAW_PATH = Path('..') / 'output' / 'combined_judge_dataset_qwen_gemma.csv'
LOW_T = 0.01
HIGH_T = 3.0
N_Q = 5

use_cols = [
    'question_id', 'judge_type', 'prompt_variant', 'judge_model', 'temperature', 'repeat_id',
    'judge_reason', 'raw_output', 'format_error',
    'model_a', 'model_b', 'human_winner', 'gold_winner',
]
raw = pd.read_csv(RAW_PATH, usecols=use_cols)

sub = raw[
    (raw['judge_type'] == 'pairwise')
    & (raw['prompt_variant'] == 'cot')
    & (raw['repeat_id'] == 0)
    & (raw['temperature'].isin([LOW_T, HIGH_T]))
] .copy()

# Keep one judge model for consistency if present.
if 'google/gemma-3-27b-it' in sub['judge_model'].astype(str).unique():
    sub = sub[sub['judge_model'].astype(str) == 'google/gemma-3-27b-it'].copy()

sub['judge_text'] = sub['judge_reason'].fillna(sub['raw_output']).fillna('').astype(str)
sub['judge_text'] = sub['judge_text'].str.strip()
sub = sub[sub['judge_text'] != ''].copy()

# Prefer non-format-error rows when duplicates exist for same (qid, temp).
sub['format_error_num'] = sub['format_error'].astype(float)
sub = sub.sort_values(['question_id', 'temperature', 'format_error_num'])
sub = sub.drop_duplicates(subset=['question_id', 'temperature'], keep='first').copy()

qids_low = set(sub.loc[sub['temperature'] == LOW_T, 'question_id'])
qids_high = set(sub.loc[sub['temperature'] == HIGH_T, 'question_id'])
common_qids = sorted(qids_low & qids_high)

# Prefer questions already in rescued explorer set to align with this notebook context.
rescued_qids = [q for q in df['question_id'].dropna().astype(int).unique().tolist() if q in common_qids]
selected_qids = rescued_qids[:N_Q]
if len(selected_qids) < N_Q:
    fill = [q for q in common_qids if q not in selected_qids][: (N_Q - len(selected_qids))]
    selected_qids = selected_qids + fill

pair_df = sub[sub['question_id'].isin(selected_qids)].copy()
pair_df['temp_label'] = np.where(pair_df['temperature'] == LOW_T, f'Low T={LOW_T}', f'High T={HIGH_T}')
pair_df = pair_df.sort_values(['question_id', 'temperature']).reset_index(drop=True)

if len(pair_df) == 0:
    raise ValueError('No valid low/high temperature pairs found.')

# Better embedding model than MiniLM.
emb_model_name = 'sentence-transformers/all-mpnet-base-v2'
try:
    from sentence_transformers import SentenceTransformer
    emb_model = SentenceTransformer(emb_model_name)
    X = emb_model.encode(pair_df['judge_text'].tolist(), show_progress_bar=False, normalize_embeddings=True)
    emb_method = emb_model_name
except Exception as e:
    # Safe fallback.
    from sentence_transformers import SentenceTransformer
    fallback_name = 'sentence-transformers/all-MiniLM-L6-v2'
    emb_model = SentenceTransformer(fallback_name)
    X = emb_model.encode(pair_df['judge_text'].tolist(), show_progress_bar=False, normalize_embeddings=True)
    emb_method = f'{fallback_name} (fallback: {e})'

pca = PCA(n_components=2, random_state=42)
X2 = pca.fit_transform(X)
pair_df['pc1'] = X2[:, 0]
pair_df['pc2'] = X2[:, 1]

fig = px.scatter(
    pair_df,
    x='pc1',
    y='pc2',
    color=pair_df['question_id'].astype(str),
    symbol='temp_label',
    hover_data=['question_id', 'temp_label', 'model_a', 'model_b', 'gold_winner', 'human_winner'],
    title=f'PCA: Low vs High Temperature Judge Texts (n_questions={len(selected_qids)})',
    height=560,
    template='plotly_white',
)
fig.update_traces(marker=dict(size=11, opacity=0.9, line=dict(width=0.6, color='white')))
fig.show()

# Pairwise low-high metrics per question.
pivot = pair_df.pivot_table(index='question_id', columns='temp_label', values='judge_text', aggfunc='first')
q_metrics = []
for q in selected_qids:
    d_q = pair_df[pair_df['question_id'] == q].copy()
    if len(d_q) != 2:
        continue
    idx = d_q.index.to_list()
    i0, i1 = idx[0], idx[1]
    dist = float(cosine_distances(X[i0:i0+1], X[i1:i1+1])[0, 0])
    sim = 1.0 - dist
    q_metrics.append({'question_id': q, 'cosine_similarity_low_high': sim, 'cosine_distance_low_high': dist})

q_metrics_df = pd.DataFrame(q_metrics).sort_values('cosine_similarity_low_high', ascending=False).reset_index(drop=True)

# Overall metrics for this low-vs-high comparison setup.
low_mask = pair_df['temp_label'].str.startswith('Low').to_numpy()
high_mask = pair_df['temp_label'].str.startswith('High').to_numpy()
same_q_dist = []
cross_q_dist = []
for i in range(len(pair_df)):
    for j in range(i + 1, len(pair_df)):
        dij = float(cosine_distances(X[i:i+1], X[j:j+1])[0, 0])
        same_q = pair_df.iloc[i]['question_id'] == pair_df.iloc[j]['question_id']
        if same_q:
            same_q_dist.append(dij)
        else:
            cross_q_dist.append(dij)

global_metrics = [
    {'Metric': 'Embedding_Method', 'Value': emb_method},
    {'Metric': 'N_questions', 'Value': int(len(selected_qids))},
    {'Metric': 'N_points_low_high', 'Value': int(len(pair_df))},
    {'Metric': 'PCA_explained_var_PC1', 'Value': float(pca.explained_variance_ratio_[0])},
    {'Metric': 'PCA_explained_var_PC2', 'Value': float(pca.explained_variance_ratio_[1])},
    {'Metric': 'PCA_explained_var_PC1_PC2_sum', 'Value': float(pca.explained_variance_ratio_[:2].sum())},
    {'Metric': 'Mean_cosine_similarity_low_high', 'Value': float(q_metrics_df['cosine_similarity_low_high'].mean()) if len(q_metrics_df) else np.nan},
    {'Metric': 'Std_cosine_similarity_low_high', 'Value': float(q_metrics_df['cosine_similarity_low_high'].std(ddof=1)) if len(q_metrics_df) > 1 else np.nan},
    {'Metric': 'Mean_same_question_cosine_distance', 'Value': float(np.mean(same_q_dist)) if same_q_dist else np.nan},
    {'Metric': 'Mean_cross_question_cosine_distance', 'Value': float(np.mean(cross_q_dist)) if cross_q_dist else np.nan},
    {'Metric': 'Cross_minus_Same_distance_gap', 'Value': (float(np.mean(cross_q_dist)) - float(np.mean(same_q_dist))) if (same_q_dist and cross_q_dist) else np.nan},
]

if len(pair_df) >= 4:
    try:
        global_metrics.append({
            'Metric': 'Silhouette_temp_label_cosine',
            'Value': float(silhouette_score(X, pair_df['temp_label'], metric='cosine'))
        })
    except Exception:
        global_metrics.append({'Metric': 'Silhouette_temp_label_cosine', 'Value': np.nan})

metrics_df = pd.DataFrame(global_metrics)
display(metrics_df)
display(q_metrics_df)

print('Selected question_ids:', selected_qids)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,Metric,Value
0,Embedding_Method,sentence-transformers/all-mpnet-base-v2
1,N_questions,5
2,N_points_low_high,10
3,PCA_explained_var_PC1,0.246908
4,PCA_explained_var_PC2,0.171307
5,PCA_explained_var_PC1_PC2_sum,0.418215
6,Mean_cosine_similarity_low_high,0.607469
7,Std_cosine_similarity_low_high,0.073135
8,Mean_same_question_cosine_distance,0.392531
9,Mean_cross_question_cosine_distance,0.557199


,question_id,cosine_similarity_low_high,cosine_distance_low_high
0,467,0.668673,0.331327
1,444,0.648130,0.351870
2,229,0.630259,0.369741
3,48,0.607219,0.392781
4,386,0.483066,0.516934


Selected question_ids: [229, 467, 386, 48, 444]


## Semantic Embedding PCA Analysis

This section builds sentence embeddings from combined response/reference text, projects them to 2D with PCA, and reports quantitative cluster/separation metrics.

## Usage Tips

1. Start with prompt filter to isolate baseline or cot.
2. Use rescue filter to focus on ensemble-only or single-only recovery.
3. Enter keyword (for example, theorem, physics, legal) to inspect domain slices.
4. Click any point in scatter plot, or move Sample slider, to inspect one sample in detail.